# 001 — Text Classification

Two approaches compared — TF-IDF + Logistic Regression vs. sentence-transformer embeddings + LightGBM.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained in this notebook — no external library imports from a shared `src/` package.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Data loading
feature_paths: list[str] = []          # local or gs:// URIs; empty -> synthetic
text_col: str = "text"
target_col: str = "label"

# Splitting
test_size: float = 0.2
random_state: int = 42

# TF-IDF
tfidf_max_features: int = 10000

# Embeddings
embedding_model: str = "all-MiniLM-L6-v2"
embedding_batch_size: int = 64

# Optuna
optuna_n_trials: int = 20
optuna_timeout_s: int | None = None

# Outputs
metrics_json_path: str = "outputs/metrics/metrics.json"
model_output_path: str = "outputs/models/model_lgbm.lgb"
plots_dir: str = "outputs/plots"
executed_notebook_path: str | None = None

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path
import re

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
)
from sklearn.decomposition import PCA

import lightgbm as lgb
import optuna
from sentence_transformers import SentenceTransformer

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Create output dirs
for _d in [plots_dir, "outputs/runs", "outputs/models", "outputs/metrics"]:
    Path(_d).mkdir(parents=True, exist_ok=True)

RUN_START = datetime.now(timezone.utc)
print(f"Run started at: {RUN_START.isoformat()}")

## 1 — Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Synthetic text generation or load from paths
# ---------------------------------------------------------------------------

categories = {
    "sports": ["football", "basketball", "player", "team", "game", "score", "championship", "athlete", "coach", "stadium", "tournament", "season", "goal", "referee", "league"],
    "technology": ["software", "algorithm", "data", "computer", "network", "digital", "innovation", "startup", "cloud", "programming", "artificial", "intelligence", "device", "internet", "processor"],
    "politics": ["government", "election", "policy", "senator", "vote", "congress", "democracy", "legislation", "president", "campaign", "debate", "law", "party", "candidate", "reform"],
    "entertainment": ["movie", "music", "actor", "celebrity", "concert", "festival", "award", "streaming", "director", "album", "premiere", "performance", "studio", "audience", "blockbuster"],
}
noise_words = ["the", "and", "is", "it", "was", "has", "this", "that", "with", "for"]

if not feature_paths:
    print("No feature_paths provided — generating synthetic multi-topic text dataset.")
    rng = np.random.default_rng(random_state)
    texts = []
    labels = []
    n_per_class = 500
    cat_names = list(categories.keys())

    for cat, vocab in categories.items():
        other_cats = [c for c in cat_names if c != cat]
        for _ in range(n_per_class):
            n_words = rng.integers(8, 16)
            chosen = list(rng.choice(vocab, size=n_words, replace=True))
            # Add some noise words
            n_noise = rng.integers(1, 4)
            chosen += list(rng.choice(noise_words, size=n_noise, replace=True))
            # 20% chance of mixing 1-2 words from another category
            if rng.random() < 0.2:
                other_cat = cat_names[rng.integers(0, len(other_cats))]
                other_vocab = categories[other_cat]
                n_cross = rng.integers(1, 3)
                chosen += list(rng.choice(other_vocab, size=n_cross, replace=True))
            rng.shuffle(chosen)
            texts.append(" ".join(chosen))
            labels.append(cat)

    df = pl.DataFrame({text_col: texts, target_col: labels})
    print(f"Synthetic dataset generated: {len(df):,} rows, {len(cat_names)} classes.")
else:
    frames = []
    for path in feature_paths:
        if path.endswith(".parquet"):
            frames.append(pl.read_parquet(path))
        elif path.endswith(".csv"):
            frames.append(pl.read_csv(path))
        else:
            raise ValueError(f"Unsupported file format: {path}")
    df = pl.concat(frames)
    print(f"Loaded {len(df):,} rows from {len(feature_paths)} file(s).")

print("\nClass distribution:")
print(df[target_col].value_counts().sort(target_col))

## 2 — EDA

In [ ]:
# ---------------------------------------------------------------------------
# Class distribution plot
# ---------------------------------------------------------------------------
class_counts = df[target_col].value_counts().sort(target_col)
class_names = class_counts[target_col].to_list()
class_cnts = class_counts["count"].to_list()
total = sum(class_cnts)

print("Class distribution:")
for name, cnt in zip(class_names, class_cnts):
    print(f"  {name:20s}: {cnt:5d} ({100 * cnt / total:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(class_names, class_cnts, color="steelblue", edgecolor="white")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_title("Class Distribution")
for i, cnt in enumerate(class_cnts):
    ax.text(i, cnt + total * 0.005, str(cnt), ha="center", va="bottom", fontsize=10)
plt.tight_layout()
save_path = Path(plots_dir) / "eda_class_dist.png"
fig.savefig(save_path, dpi=120)
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Text length analysis
# ---------------------------------------------------------------------------
pd_df = df.to_pandas()
pd_df["word_count"] = pd_df[text_col].str.split().str.len()
pd_df["char_count"] = pd_df[text_col].str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram: word count
axes[0].hist(pd_df["word_count"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Word Count Distribution")

# Histogram: char count
axes[1].hist(pd_df["char_count"], bins=30, color="darkorange", edgecolor="white")
axes[1].set_xlabel("Character Count")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Character Count Distribution")

# Box plot: word count by class
uniq_classes = sorted(pd_df[target_col].unique())
boxdata = [pd_df.loc[pd_df[target_col] == c, "word_count"].values for c in uniq_classes]
axes[2].boxplot(boxdata, labels=uniq_classes, patch_artist=True)
axes[2].set_xlabel("Class")
axes[2].set_ylabel("Word Count")
axes[2].set_title("Word Count by Class")
plt.setp(axes[2].get_xticklabels(), rotation=20, ha="right")

plt.tight_layout()
save_path = Path(plots_dir) / "eda_text_lengths.png"
fig.savefig(save_path, dpi=120)
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Top TF-IDF terms per class
# ---------------------------------------------------------------------------
n_top = 15
uniq_classes = sorted(pd_df[target_col].unique())
n_classes = len(uniq_classes)
ncols = min(n_classes, 4)
nrows = (n_classes + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).flatten()

for idx, cls in enumerate(uniq_classes):
    class_texts = pd_df.loc[pd_df[target_col] == cls, text_col].tolist()
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))
    tfidf_mat = vec.fit_transform(class_texts)
    mean_scores = np.asarray(tfidf_mat.mean(axis=0)).flatten()
    top_idx = mean_scores.argsort()[-n_top:][::-1]
    top_terms = [vec.get_feature_names_out()[i] for i in top_idx]
    top_scores = mean_scores[top_idx]

    ax = axes[idx]
    ax.barh(top_terms[::-1], top_scores[::-1], color="steelblue")
    ax.set_title(f"Top {n_top} TF-IDF Terms: {cls}")
    ax.set_xlabel("Mean TF-IDF Score")

# Hide unused subplots
for j in range(n_classes, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
save_path = Path(plots_dir) / "eda_top_tfidf_terms.png"
fig.savefig(save_path, dpi=120)
plt.show()
print(f"Saved: {save_path}")

## 3 — Train / Test Split

In [ ]:
# ---------------------------------------------------------------------------
# Stratified train/test split
# ---------------------------------------------------------------------------
X = pd_df[text_col].tolist()
y = pd_df[target_col].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

print(f"Train size: {len(X_train):,}")
print(f"Test size:  {len(X_test):,}")
print()
print("Train class distribution:")
train_ser = pd.Series(y_train)
for cls, cnt in train_ser.value_counts().sort_index().items():
    print(f"  {cls:20s}: {cnt:5d} ({100 * cnt / len(y_train):.1f}%)")
print()
print("Test class distribution:")
test_ser = pd.Series(y_test)
for cls, cnt in test_ser.value_counts().sort_index().items():
    print(f"  {cls:20s}: {cnt:5d} ({100 * cnt / len(y_test):.1f}%)")

## 4 — TF-IDF + Logistic Regression

In [ ]:
# ---------------------------------------------------------------------------
# Build pipeline and GridSearchCV
# ---------------------------------------------------------------------------
tfidf_lr_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=tfidf_max_features,
        ngram_range=(1, 2),
        sublinear_tf=True,
    )),
    ("lr", LogisticRegression(
        max_iter=1000,
        random_state=random_state,
    )),
])

param_grid = {
    "lr__C": [0.01, 0.1, 1.0, 10.0],
    "lr__solver": ["lbfgs", "liblinear"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
grid_search = GridSearchCV(
    tfidf_lr_pipe,
    param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True,
    verbose=0,
)
grid_search.fit(X_train, y_train)

print(f"Best params:    {grid_search.best_params_}")
print(f"Best CV macro-F1: {grid_search.best_score_:.4f}")

best_tfidf_lr = grid_search.best_estimator_

In [ ]:
# ---------------------------------------------------------------------------
# Evaluate TF-IDF + LR on test set
# ---------------------------------------------------------------------------
y_pred_tfidf = best_tfidf_lr.predict(X_test)
y_proba_tfidf = best_tfidf_lr.predict_proba(X_test)

uniq_labels = sorted(set(y_test))
n_classes = len(uniq_labels)

print("Classification Report (TF-IDF + LR):")
print(classification_report(y_test, y_pred_tfidf))

acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
f1_macro_tfidf = f1_score(y_test, y_pred_tfidf, average="macro")
f1_weighted_tfidf = f1_score(y_test, y_pred_tfidf, average="weighted")

if n_classes == 2:
    auc_tfidf = roc_auc_score(y_test, y_proba_tfidf[:, 1])
else:
    auc_tfidf = roc_auc_score(
        y_test, y_proba_tfidf,
        multi_class="ovr", average="macro",
        labels=uniq_labels,
    )

print(f"Accuracy:      {acc_tfidf:.4f}")
print(f"Macro-F1:      {f1_macro_tfidf:.4f}")
print(f"Weighted-F1:   {f1_weighted_tfidf:.4f}")
print(f"ROC-AUC (OvR): {auc_tfidf:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Confusion matrix — TF-IDF + LR
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_tfidf,
    display_labels=uniq_labels,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("Confusion Matrix — TF-IDF + Logistic Regression")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
save_path = Path(plots_dir) / "tfidf_confusion_matrix.png"
fig.savefig(save_path, dpi=120)
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Top TF-IDF features by LR coefficient magnitude
# ---------------------------------------------------------------------------
lr_model = best_tfidf_lr.named_steps["lr"]
vectorizer = best_tfidf_lr.named_steps["tfidf"]
feature_names = vectorizer.get_feature_names_out()

n_top_feat = 15
classes = lr_model.classes_
n_cls = len(classes)
ncols = min(n_cls, 4)
nrows = (n_cls + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).flatten()

for idx, cls in enumerate(classes):
    if n_cls == 2:
        # binary: only one set of coefs
        coef = lr_model.coef_[0] if idx == 0 else -lr_model.coef_[0]
    else:
        coef = lr_model.coef_[idx]
    top_idx = np.argsort(coef)[-n_top_feat:]
    top_feats = feature_names[top_idx]
    top_coefs = coef[top_idx]

    ax = axes[idx]
    colors = ["tomato" if c < 0 else "steelblue" for c in top_coefs]
    ax.barh(top_feats, top_coefs, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Top Features: {cls}")
    ax.set_xlabel("Coefficient")

for j in range(n_cls, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Top TF-IDF Features by LR Coefficient", fontsize=13, y=1.01)
plt.tight_layout()
save_path = Path(plots_dir) / "tfidf_top_features.png"
fig.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

## 5 — Sentence Embeddings + LightGBM

In [ ]:
# ---------------------------------------------------------------------------
# Load SentenceTransformer and encode texts
# ---------------------------------------------------------------------------
print(f"Loading SentenceTransformer model: {embedding_model}")
print("Note: first run will download the model (~90MB for all-MiniLM-L6-v2).")
st_model = SentenceTransformer(embedding_model)
print(f"Model loaded. Max sequence length: {st_model.max_seq_length}")

print("Encoding training texts...")
X_train_emb = st_model.encode(
    X_train,
    batch_size=embedding_batch_size,
    show_progress_bar=False,
)
print("Encoding test texts...")
X_test_emb = st_model.encode(
    X_test,
    batch_size=embedding_batch_size,
    show_progress_bar=False,
)

print(f"Train embedding shape: {X_train_emb.shape}")
print(f"Test embedding shape:  {X_test_emb.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# PCA 2D visualization of embeddings
# ---------------------------------------------------------------------------
print("Fitting PCA on train embeddings...")
pca = PCA(n_components=2, random_state=random_state)
X_train_pca = pca.fit_transform(X_train_emb)
explained = pca.explained_variance_ratio_
print(f"PCA explained variance: PC1={explained[0]:.3f}, PC2={explained[1]:.3f}, total={sum(explained):.3f}")

uniq_cls = sorted(set(y_train))
cmap = plt.get_cmap("tab10")
cls_to_color = {c: cmap(i) for i, c in enumerate(uniq_cls)}

fig, ax = plt.subplots(figsize=(9, 7))
y_train_arr = np.array(y_train)
for cls in uniq_cls:
    mask = y_train_arr == cls
    ax.scatter(
        X_train_pca[mask, 0], X_train_pca[mask, 1],
        label=cls, alpha=0.4, s=15,
        color=cls_to_color[cls],
    )

ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}% var)")
ax.set_title("PCA 2D — Sentence Embeddings (train set)")
ax.legend(markerscale=2, loc="best")
plt.tight_layout()
save_path = Path(plots_dir) / "embedding_pca.png"
fig.savefig(save_path, dpi=120)
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Optuna tuning for LightGBM
# ---------------------------------------------------------------------------
def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "random_state": random_state,
        "verbosity": -1,
        "n_jobs": -1,
    }
    clf = lgb.LGBMClassifier(**params)
    cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    scores = cross_val_score(
        clf, X_train_emb, y_train,
        cv=cv_inner, scoring="f1_macro", n_jobs=-1,
    )
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(
    objective,
    n_trials=optuna_n_trials,
    timeout=optuna_timeout_s,
    show_progress_bar=False,
)

best_lgbm_params = study.best_params
best_lgbm_params["random_state"] = random_state
best_lgbm_params["verbosity"] = -1
best_lgbm_params["n_jobs"] = -1

print(f"Best Optuna CV macro-F1: {study.best_value:.4f}")
print(f"Best params: {best_lgbm_params}")

In [ ]:
# ---------------------------------------------------------------------------
# Train final LGBMClassifier and evaluate
# ---------------------------------------------------------------------------
final_lgbm = lgb.LGBMClassifier(**best_lgbm_params)
final_lgbm.fit(X_train_emb, y_train)

y_pred_lgbm = final_lgbm.predict(X_test_emb)
y_proba_lgbm = final_lgbm.predict_proba(X_test_emb)

print("Classification Report (Embeddings + LightGBM):")
print(classification_report(y_test, y_pred_lgbm))

acc_lgbm = accuracy_score(y_test, y_pred_lgbm)
f1_macro_lgbm = f1_score(y_test, y_pred_lgbm, average="macro")
f1_weighted_lgbm = f1_score(y_test, y_pred_lgbm, average="weighted")

if n_classes == 2:
    auc_lgbm = roc_auc_score(y_test, y_proba_lgbm[:, 1])
else:
    auc_lgbm = roc_auc_score(
        y_test, y_proba_lgbm,
        multi_class="ovr", average="macro",
        labels=uniq_labels,
    )

print(f"Accuracy:      {acc_lgbm:.4f}")
print(f"Macro-F1:      {f1_macro_lgbm:.4f}")
print(f"Weighted-F1:   {f1_weighted_lgbm:.4f}")
print(f"ROC-AUC (OvR): {auc_lgbm:.4f}")

# Save model
Path(model_output_path).parent.mkdir(parents=True, exist_ok=True)
final_lgbm.booster_.save_model(model_output_path)
print(f"\nLightGBM model saved to: {model_output_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Confusion matrix — LightGBM
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lgbm,
    display_labels=uniq_labels,
    cmap="Greens",
    ax=ax,
    colorbar=False,
)
ax.set_title("Confusion Matrix — Embeddings + LightGBM")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
save_path = Path(plots_dir) / "lgbm_confusion_matrix.png"
fig.savefig(save_path, dpi=120)
plt.show()
print(f"Saved: {save_path}")

## 6 — Model Comparison

In [ ]:
# ---------------------------------------------------------------------------
# Side-by-side comparison table
# ---------------------------------------------------------------------------
comparison = pd.DataFrame({
    "Model": ["TF-IDF + LogReg", "Embeddings + LightGBM"],
    "Accuracy": [acc_tfidf, acc_lgbm],
    "Macro-F1": [f1_macro_tfidf, f1_macro_lgbm],
    "Weighted-F1": [f1_weighted_tfidf, f1_weighted_lgbm],
    "ROC-AUC (OvR)": [auc_tfidf, auc_lgbm],
})
comparison = comparison.set_index("Model")
print(comparison.round(4).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Bar chart comparison
# ---------------------------------------------------------------------------
metric_cols = ["Accuracy", "Macro-F1", "Weighted-F1", "ROC-AUC (OvR)"]
model_names = comparison.index.tolist()

x = np.arange(len(metric_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
for i, model in enumerate(model_names):
    vals = [comparison.loc[model, m] for m in metric_cols]
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=model)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{val:.3f}",
            ha="center", va="bottom", fontsize=8,
        )

ax.set_xticks(x)
ax.set_xticklabels(metric_cols)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — TF-IDF+LR vs. Embeddings+LightGBM")
ax.legend(loc="upper right")
plt.tight_layout()
save_path = Path(plots_dir) / "model_comparison.png"
fig.savefig(save_path, dpi=120)
plt.show()
print(f"Saved: {save_path}")

## 7 — Metrics JSON Output

In [ ]:
# ---------------------------------------------------------------------------
# Write metrics JSON
# ---------------------------------------------------------------------------
RUN_END = datetime.now(timezone.utc)

metrics_payload = {
    "run_metadata": {
        "timestamp": RUN_START.isoformat(),
        "duration_s": (RUN_END - RUN_START).total_seconds(),
        "text_col": text_col,
        "target_col": target_col,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "n_classes": n_classes,
        "tfidf_max_features": tfidf_max_features,
        "embedding_model": embedding_model,
        "optuna_n_trials": optuna_n_trials,
    },
    "models": {
        "tfidf_logreg": {
            "accuracy": acc_tfidf,
            "macro_f1": f1_macro_tfidf,
            "weighted_f1": f1_weighted_tfidf,
            "roc_auc_ovr": auc_tfidf,
            "best_cv_macro_f1": grid_search.best_score_,
            "best_params": grid_search.best_params_,
        },
        "embeddings_lgbm": {
            "accuracy": acc_lgbm,
            "macro_f1": f1_macro_lgbm,
            "weighted_f1": f1_weighted_lgbm,
            "roc_auc_ovr": auc_lgbm,
            "best_cv_macro_f1": study.best_value,
            "best_params": best_lgbm_params,
            "model_path": model_output_path,
        },
    },
}

Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)
with open(metrics_json_path, "w") as f:
    json.dump(metrics_payload, f, indent=2, default=str)

print(f"Metrics saved to: {metrics_json_path}")

## 8 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
if f1_macro_lgbm > f1_macro_tfidf:
    winner = "Embeddings + LightGBM"
    delta = f1_macro_lgbm - f1_macro_tfidf
else:
    winner = "TF-IDF + Logistic Regression"
    delta = f1_macro_tfidf - f1_macro_lgbm

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print()
print(f"{'Metric':<22} {'TF-IDF+LR':>12} {'Emb+LGBM':>12}")
print("-" * 48)
print(f"{'Accuracy':<22} {acc_tfidf:>12.4f} {acc_lgbm:>12.4f}")
print(f"{'Macro-F1':<22} {f1_macro_tfidf:>12.4f} {f1_macro_lgbm:>12.4f}")
print(f"{'Weighted-F1':<22} {f1_weighted_tfidf:>12.4f} {f1_weighted_lgbm:>12.4f}")
print(f"{'ROC-AUC (OvR)':<22} {auc_tfidf:>12.4f} {auc_lgbm:>12.4f}")
print()
print(f"Winner on Macro-F1: {winner} (+{delta:.4f})")
print()
print(f"LightGBM model saved to : {model_output_path}")
print(f"Metrics JSON saved to   : {metrics_json_path}")
print(f"Plots saved to          : {plots_dir}/")
print()
print(f"Run duration: {(RUN_END - RUN_START).total_seconds():.1f}s")